Link: [Ultimate RAG Bootcamp Using Langchain,LangGraph & Langsmith](https://www.udemy.com/course/ultimate-rag-bootcamp-using-langchainlanggraph-langsmith)

### HyDE
🧠 What is HyDe?

HyDE (Hypothetical Document Embeddings) is a retrieval technique where, instead of embedding the user’s query directly, you first generate a hypothetical answer (document) to the query using an LLM — and then embed that hypothetical document to search your vector store.

➡️ HyDE bridges the gap between user intent and relevant content, especially when:

1. Queries are short
2. Language mismatch between query and documents
3.You want to retrieve based on answer content, not question words

In [1]:
# ============================================================================
# PART 1: MANUAL HyDE IMPLEMENTATION
# ============================================================================
# 
# First, we'll implement HyDE manually to understand how it works.
# Then we'll use LangChain's built-in HypotheticalDocumentEmbedder.
# ============================================================================

# ============================================================================
# STEP 0: Import Required Libraries
# ============================================================================

from langchain_community.document_loaders import WikipediaLoader   # Load Wikipedia articles
from langchain.text_splitter import RecursiveCharacterTextSplitter  # Split text
from langchain_huggingface import HuggingFaceEmbeddings             # Local embeddings
from langchain.vectorstores import Chroma                           # Persistent vector store

/Users/sourav.banerjee/Documents/Codebases/2. AI ENGINEERING/RAG_Demystified/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ============================================================================
# STEP 1: Load and Chunk the Dataset
# ============================================================================
# 
# For this example, we'll load Wikipedia articles about Steve Jobs.
# WikipediaLoader fetches real Wikipedia content for our knowledge base.
#
# Chunking Strategy:
# - chunk_size=300: Small chunks for precise retrieval
# - chunk_overlap=100: Significant overlap to maintain context
#   (33% overlap helps ensure important information isn't split)
# ============================================================================

chunk_size = 300
chunk_overlap = 100

# Load Wikipedia articles about Steve Jobs
print("📥 Loading Wikipedia articles...")
loader = WikipediaLoader(query="Steve Jobs", load_max_docs=5)
documents = loader.load()
print(f"   Loaded {len(documents)} documents")

# Split into smaller chunks for better retrieval
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size, 
    chunk_overlap=chunk_overlap
)
docs = text_splitter.split_documents(documents=documents)
print(f"📦 Created {len(docs)} chunks")
print(f"\n📄 Sample chunk:\n{docs[0].page_content[:200]}...")
docs

📥 Loading Wikipedia articles...
   Loaded 4 documents
📦 Created 73 chunks

📄 Sample chunk:
Steven Paul Jobs (February 24, 1955 – October 5, 2011) was an American businessman, inventor, and investor best known for co-founding the technology company Apple Inc. Jobs was also the founder of NeX...


[Document(metadata={'title': 'Steve Jobs', 'summary': 'Steven Paul Jobs (February 24, 1955 – October 5, 2011) was an American businessman, inventor, and investor best known for co-founding the technology company Apple Inc. Jobs was also the founder of NeXT and chairman and majority shareholder of Pixar. He was a pioneer of the personal computer revolution of the 1970s and 1980s, along with his early business partner and fellow Apple co-founder Steve Wozniak.\nJobs was born in San Francisco in 1955 and adopted shortly afterwards. He attended Reed College in 1972 before withdrawing that same year. In 1974, he traveled through India, seeking enlightenment before later studying Zen Buddhism. He and Wozniak co-founded Apple in 1976 to further develop and sell Wozniak\'s Apple I personal computer. Together, the duo gained fame and wealth a year later with production and sale of the Apple II, one of the first highly successful mass-produced microcomputers. \nJobs saw the commercial potential 

In [6]:
# ============================================================================
# STEP 2: Build the Vector Store
# ============================================================================
# 
# We use FAISS for fast similarity search.
# The embeddings are generated using a local HuggingFace model.
#
# Note: This is our "standard" vector store - we'll search it using
# the hypothetical document embeddings instead of query embeddings.
# ============================================================================

from langchain.vectorstores import FAISS

# Initialize embedding model (runs locally, no API costs)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Create FAISS vector store from documents
vectorstore = FAISS.from_documents(docs, embeddings)
print(f"✅ Created FAISS vector store with {len(docs)} vectors")

✅ Created FAISS vector store with 73 vectors


In [7]:
# 3. Set up the LLM you’ll use to generate hypothetical answers
import os
from dotenv import load_dotenv
load_dotenv()
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
llm = init_chat_model("groq:llama-3.1-8b-instant")

In [8]:
# ============================================================================
# STEP 4: Create Chroma Vector Store with Persistence
# ============================================================================
# 
# We also create a Chroma vector store for comparison.
# Chroma supports persistence - the vectors are saved to disk.
#
# Key differences from FAISS:
# - Chroma: Persistent storage, built-in filtering, good for production
# - FAISS: In-memory by default, very fast, good for development
# ============================================================================

from langchain.vectorstores import Chroma

# Create Chroma vector store with persistence
db = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory="output/steve_jobs_for_hyde.db"
)

# Create a standard retriever for comparison
base_retriever = db.as_retriever(search_kwargs={"k": 5})
print("✅ Chroma vector store created and persisted")

✅ Chroma vector store created and persisted


In [9]:
# ============================================================================
# STEP 5: Create the HyDE Document Generator (Manual Implementation)
# ============================================================================
# 
# This function implements the core HyDE logic:
# 1. Takes a user query as input
# 2. Generates a hypothetical answer using the LLM
# 3. Returns the hypothetical document (which will be embedded for search)
#
# The prompt asks the LLM to act as an expert writing a detailed explanation.
# This generates content that is semantically similar to actual documents.
#
# 🔑 Key Insight: The hypothetical answer doesn't need to be factually correct!
#    It just needs to be semantically similar to where the real answer would be.
# ============================================================================

from langchain_core.output_parsers import StrOutputParser
from langchain.prompts.chat import SystemMessagePromptTemplate, ChatPromptTemplate

def get_hyde_doc(query):
    """
    Generate a hypothetical document for the given query.
    
    This is the core of HyDE: we generate what the answer MIGHT look like,
    then use that to search for actual documents with similar content.
    
    Args:
        query: The user's question
        
    Returns:
        A hypothetical answer/document generated by the LLM
    """
    # Prompt that encourages detailed, expert-like content
    template = """Imagine you are an expert writing a detailed explanation on the topic: '{query}'
    create a hypothetical answer for the topic"""

    # Create the prompt
    system_message_prompt = SystemMessagePromptTemplate.from_template(template=template)
    chat_prompt = ChatPromptTemplate.from_messages([system_message_prompt])
    messages = chat_prompt.format_prompt(query=query).to_messages()
    
    print(f"📤 Generating hypothetical document for: '{query}'")
    print(f"   Prompt: {messages}")
    
    # Generate hypothetical answer
    response = llm.invoke(messages)
    hypo_doc = response.content
    
    return hypo_doc

print("✅ HyDE document generator function created")

✅ HyDE document generator function created


In [10]:
# ============================================================================
# 🧪 TEST: Generate a Hypothetical Document
# ============================================================================
# 
# Let's see what a hypothetical document looks like!
# Notice how the LLM generates a detailed answer that:
# - Contains relevant terminology
# - Has similar structure to actual documents
# - Captures the semantic essence of the topic
#
# Even if some facts are wrong, the SEMANTIC CONTENT will match real docs!
# ============================================================================

query = 'When was Steve Jobs fired from Apple?'

print("="*70)
print("🧪 TESTING HYDE DOCUMENT GENERATION")
print("="*70)
print(f"\n📝 Original Query: {query}\n")

hypothetical_doc = get_hyde_doc(query=query)

print("\n📄 GENERATED HYPOTHETICAL DOCUMENT:")
print("-"*70)
print(hypothetical_doc)

🧪 TESTING HYDE DOCUMENT GENERATION

📝 Original Query: When was Steve Jobs fired from Apple?

📤 Generating hypothetical document for: 'When was Steve Jobs fired from Apple?'
   Prompt: [SystemMessage(content="Imagine you are an expert writing a detailed explanation on the topic: 'When was Steve Jobs fired from Apple?'\n    create a hypothetical answer for the topic", additional_kwargs={}, response_metadata={})]


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



📄 GENERATED HYPOTHETICAL DOCUMENT:
----------------------------------------------------------------------
**The Firing of Steve Jobs from Apple: A Turning Point in the Company's History**

Steve Jobs, the co-founder and visionary behind the revolutionary technology company Apple, was fired from his position as CEO in 1985. This pivotal moment in Apple's history marked a significant shift in the company's trajectory and had far-reaching consequences for both Jobs and the company.

**The Circumstances Leading to Jobs' Departure**

After co-founding Apple in 1976, Jobs and his business partner, Steve Wozniak, developed the Apple I and Apple II computers, which catapulted the company to success. However, as the company grew, Jobs and Wozniak's differing personalities and work styles began to create tension. In 1980, Jobs left Apple after a power struggle with John Sculley, whom he had hired as CEO in 1983.

**The Board of Directors' Decision**

In May 1985, the Apple Board of Directors, l

In [11]:
# ============================================================================
# STEP 6: Use HyDE for Retrieval
# ============================================================================
# 
# Now we use the hypothetical document for retrieval!
# 
# Process:
# 1. Generate hypothetical document from query
# 2. Pass hypothetical doc to retriever (which embeds it)
# 3. Retrieve documents similar to the hypothetical answer
#
# The retrieved documents should be more relevant because we're
# searching in "answer space" rather than "question space"!
# ============================================================================

print("="*70)
print("🔍 HYDE RETRIEVAL IN ACTION")
print("="*70)

# Generate hypothetical document
hypo_doc = get_hyde_doc(query)

print(f"\n✅ Hypothetical doc generated (length: {len(hypo_doc)} chars)")
print("\n📚 RETRIEVED DOCUMENTS (using hypothetical doc embedding):")
print("-"*70)

# Retrieve using the hypothetical document
matched_docs = base_retriever.invoke(hypo_doc)

for i, doc in enumerate(matched_docs, 1):
    print(f"\n📄 Document {i}:")
    print(f"   Source: {doc.metadata.get('source', 'Unknown')}")
    print(f"   Content: {doc.page_content[:200]}...")

🔍 HYDE RETRIEVAL IN ACTION
📤 Generating hypothetical document for: 'When was Steve Jobs fired from Apple?'
   Prompt: [SystemMessage(content="Imagine you are an expert writing a detailed explanation on the topic: 'When was Steve Jobs fired from Apple?'\n    create a hypothetical answer for the topic", additional_kwargs={}, response_metadata={})]

✅ Hypothetical doc generated (length: 3404 chars)

📚 RETRIEVED DOCUMENTS (using hypothetical doc embedding):
----------------------------------------------------------------------

📄 Document 1:
   Source: https://en.wikipedia.org/wiki/Steve_Jobs
   Content: In 1985, Jobs departed Apple after a long power struggle with the company's board and its then-CEO, John Sculley. That same year, Jobs took some Apple employees with him to found NeXT, a computer plat...

📄 Document 2:
   Source: https://en.wikipedia.org/wiki/Steve_Jobs_(film)
   Content: conducted by Sorkin. The film covers fourteen years in the life of Apple Inc. co-founder Steve Jobs, s

---

# PART 2: LangChain's Built-in HyDE Support

## 🔧 HypotheticalDocumentEmbedder

LangChain provides a built-in `HypotheticalDocumentEmbedder` class that:
- Wraps the HyDE logic in a reusable component
- Works as a drop-in replacement for regular embeddings
- Supports various domain-specific prompts (web search, scientific, etc.)

This is the **recommended approach** for production use!

In [ ]:
# ============================================================================
# STEP 7: Setup for LangChain's HypotheticalDocumentEmbedder
# ============================================================================
# 
# Now we'll use LangChain's built-in HyDE implementation.
# This approach is cleaner and recommended for production use.
#
# We'll load a different dataset (LangChain/CrewAI) to demonstrate
# HyDE's versatility across different domains.
# ============================================================================

from langchain.chains.hyde.base import HypotheticalDocumentEmbedder
from langchain.prompts import PromptTemplate
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains.combine_documents import create_stuff_documents_chain

# Load and split the LangChain/CrewAI dataset
print("📥 Loading LangChain/CrewAI dataset...")
loader = TextLoader("langchain_crewai_dataset.txt")
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(docs)
print(f"📦 Created {len(chunks)} chunks")

In [ ]:
# ============================================================================
# STEP 8: Set Up Base Embeddings
# ============================================================================
# 
# The HypotheticalDocumentEmbedder wraps around "base embeddings".
# It will:
# 1. Generate hypothetical document using LLM
# 2. Embed the hypothetical document using base_embeddings
#
# This allows us to use HyDE with any embedding model!
# ============================================================================

base_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
print("✅ Base embeddings initialized (all-MiniLM-L6-v2)")

## 📋 Available Prompt Keys

LangChain's `HypotheticalDocumentEmbedder` comes with pre-built prompts for different domains.
According to the official documentation and source code (`PROMPT_MAP`), the available options are:

| Prompt Key | Use Case |
|------------|----------|
| `web_search` | General web search queries |
| `sci_fact` | Scientific fact verification |
| `arguana` | Argument mining and analysis |
| `trec_covid` | COVID-19 related queries |
| `fiqa` | Financial question answering |
| `dbpedia_entity` | Entity-related queries |
| `trec_news` | News-related queries |
| `mr_tydi` | Multilingual information retrieval |

Choose the prompt key that best matches your domain for optimal hypothetical document generation!

In [ ]:
# ============================================================================
# STEP 9: Create the HypotheticalDocumentEmbedder
# ============================================================================
# 
# This is the magic! HypotheticalDocumentEmbedder:
# - Takes an LLM and base embeddings
# - Uses a domain-specific prompt (prompt_key)
# - Automatically generates hypothetical docs and embeds them
#
# prompt_key="web_search" uses a general-purpose prompt suitable
# for most web-search-like queries.
#
# 💡 This can be used as a drop-in replacement for regular embeddings!
# ============================================================================

hyde_embedding_function = HypotheticalDocumentEmbedder.from_llm(
    llm=llm,                          # LLM for generating hypothetical docs
    base_embeddings=base_embeddings,   # Embeddings for the hypothetical doc
    prompt_key="web_search"            # Domain-specific prompt template
)

print("✅ HypotheticalDocumentEmbedder created with 'web_search' prompt")

In [ ]:
# ============================================================================
# STEP 10: Create Vector Store with HyDE Embeddings
# ============================================================================
# 
# IMPORTANT: When we create the vector store, the HyDE embedder is used
# to embed the documents. This means each document's embedding is actually
# generated from a hypothetical version!
#
# At query time:
# - Query is converted to hypothetical answer
# - Hypothetical answer is embedded
# - Search finds documents with similar embeddings
#
# Note: For document indexing, you might want to use regular embeddings
# and only use HyDE for query-time. This implementation uses HyDE for both.
# ============================================================================

print("📦 Creating Chroma vector store with HyDE embeddings...")
print("   (This may take a moment as each chunk generates a hypothetical doc)")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=hyde_embedding_function,  # HyDE embeddings!
    persist_directory="output/langchain"
)

print("✅ Vector store created and persisted")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [ ]:
# ============================================================================
# STEP 11: Create the RAG Answer Generation Chain
# ============================================================================
# 
# Standard RAG chain that takes retrieved context and generates an answer.
# The HyDE magic happens in retrieval - this part is the same as any RAG!
# ============================================================================

rag_prompt = PromptTemplate.from_template("""
Use the context below to answer the question.

Context:
{context}

Question: {input}
""")

rag_chain = create_stuff_documents_chain(llm=llm, prompt=rag_prompt)
print("✅ RAG answer chain created")

In [ ]:
# ============================================================================
# STEP 12: Build the Complete HyDE RAG Pipeline
# ============================================================================
# 
# This function combines HyDE retrieval with answer generation:
# 1. Query → [HyDE embedding] → Similar documents
# 2. Documents + Query → [LLM] → Answer
#
# The vector store's similarity_search automatically uses HyDE
# because we created it with the HypotheticalDocumentEmbedder!
# ============================================================================

def hyde_rag_pipeline(query):
    """
    Execute RAG pipeline with HyDE retrieval.
    
    Args:
        query: User's question
        
    Returns:
        Generated answer based on HyDE-retrieved context
    """
    print(f"📝 Query: {query}")
    print("\n🔍 Retrieving documents using HyDE...")
    
    # This uses HyDE: query → hypothetical doc → embed → search
    matched_docs = vectorstore.similarity_search(query, k=4)
    
    print(f"\n📚 Retrieved {len(matched_docs)} documents:")
    for i, doc in enumerate(matched_docs, 1):
        print(f"   {i}. {doc.page_content[:80]}...")
    
    # Generate answer from retrieved context
    response = rag_chain.invoke({
        "input": query,
        "context": matched_docs
    })
    
    return response

print("✅ HyDE RAG pipeline ready!")

In [ ]:
# ============================================================================
# STEP 13: Run the HyDE RAG Pipeline
# ============================================================================
# 
# Let's test our HyDE-powered RAG system!
# The query will be converted to a hypothetical answer before retrieval,
# which should improve the relevance of retrieved documents.
# ============================================================================

query = "What memory modules does LangChain provide?"

print("="*70)
print("🚀 RUNNING HYDE RAG PIPELINE")
print("="*70 + "\n")

answer = hyde_rag_pipeline(query)

print("\n" + "="*70)
print("✅ FINAL ANSWER")
print("="*70)
print(answer)

[Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain offers memory modules like ConversationBufferMemory and ConversationSummaryMemory. These allow the LLM to maintain awareness of previous conversation turns or summarize long interactions to fit within token limits. (v6)'), Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain offers memory modules like ConversationBufferMemory and ConversationSummaryMemory. These allow the LLM to maintain awareness of previous conversation turns or summarize long interactions to fit within token limits. (v10)'), Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain offers memory modules like ConversationBufferMemory and ConversationSummaryMemory. These allow the LLM to maintain awareness of previous conversation turns or summarize long interactions to fit within token limits. (v2)'), Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_cont

---

# PART 3: Using Custom Prompts with HyDE

## 🎨 Custom Prompt Templates

Sometimes the built-in prompts don't fit your domain. You can create custom prompts
that generate more appropriate hypothetical documents for your use case!

In [ ]:
# ============================================================================
# STEP 14: Create HyDE with Custom Prompt
# ============================================================================
# 
# For domain-specific applications, you can provide a custom prompt
# that generates hypothetical documents tailored to your use case.
#
# Examples of custom prompts:
# - Technical documentation: "Write a technical explanation for..."
# - Legal domain: "Draft a legal response regarding..."
# - Medical: "Provide a medical explanation for..."
#
# 💡 The better your prompt matches your document style, the better
#    the retrieval will be!
# ============================================================================

from langchain.prompts import PromptTemplate

# Create a custom prompt for concise hypothetical answers
custom_prompt = PromptTemplate.from_template(
    "Generate a concise hypothetical answer for this topic: {query}"
)

# Create HyDE embedder with custom prompt
hyde_custom = HypotheticalDocumentEmbedder.from_llm(
    llm=llm,
    base_embeddings=base_embeddings,
    custom_prompt=custom_prompt  # Use custom prompt instead of prompt_key
)

print("✅ HyDE embedder created with custom prompt")
print(f"   Prompt template: '{custom_prompt.template}'")

## 📊 Summary: HyDE Benefits and Trade-offs

### Comparison: Traditional vs HyDE Retrieval

| Aspect | Traditional Retrieval | HyDE Retrieval |
|--------|----------------------|----------------|
| **Query Embedding** | Embed the question directly | Embed a hypothetical answer |
| **Semantic Space** | Question space | Answer space (closer to documents) |
| **Short Queries** | Often miss relevant docs | Expanded into detailed hypothetical |
| **Latency** | Fast (just embedding) | Slower (LLM call + embedding) |
| **Best For** | Long, detailed queries | Short, ambiguous queries |

## 🎯 Key Takeaways

1. **HyDE generates hypothetical answers** before embedding for retrieval
2. **Bridges the semantic gap** between questions and documents
3. **Works with any embedding model** via HypotheticalDocumentEmbedder
4. **Custom prompts** allow domain-specific optimization

## ⚠️ Considerations

- **Latency**: Adds an LLM call for every query
- **Cost**: More API calls for hosted LLMs
- **LLM Quality**: Hypothetical doc quality depends on LLM capability
- **Not Always Needed**: Long, detailed queries may not benefit

## 🔧 Best Practices

1. **Use domain-specific prompts** for better hypothetical docs
2. **Consider caching** for repeated or similar queries
3. **A/B test** against traditional retrieval for your use case
4. **Monitor retrieval quality** to ensure HyDE is helping

## 🔗 Related Techniques

- **Query Expansion**: Add synonyms and related terms (Notebook 1)
- **Query Decomposition**: Break complex queries into parts (Notebook 2)
- **Hybrid Search**: Combine dense and sparse retrieval
- **Reranking**: Use cross-encoders to reorder results
